In [ ]:
# EgoHands benchmark for SpectatorAI: ONE cell, A100, ~30-40 min. Prints an accuracy table at the end.
# EgoHands (Bambach et al., ICCV 2015): 48 egocentric videos, 100 labelled frames each, hands labelled as
# "my left / my right" (the wearer) vs "your left / your right" (the other person). This tests exactly our claim:
# keep the wearer's hands, reject other people's, and get left/right right.
import os, sys, re, glob, json, shutil, subprocess, zipfile, pickle, time
TEST = os.environ.get("EGO_TEST") == "1"           # local plumbing test only
N_VIDEOS = int(os.environ.get("EGO_N", "16"))      # 16 of 48 videos (~10 min of GPU); 48 = all (~30 min)
WORKERS = 4
FPS = 5                                            # labelled frames are ~1 s apart; played back as a 5 fps clip
IOU = 0.3
t_start = time.time()
sh = lambda c: subprocess.run(c, shell=True, check=True)          # noqa: E731

if not TEST:
    # 1) install (same as the PS4 notebook)
    sh('pip -q install ultralytics "smplx==0.1.28" timm einops roma scikit-image huggingface_hub pyyaml setuptools wheel')
    sh("pip -q install --no-deps git+https://github.com/warmshao/WiLoR-mini")
    sh("pip -q install --no-build-isolation --no-deps git+https://github.com/mattloper/chumpy")
    sh("pip -q install git+https://github.com/cvg/GeoCalib")
    import importlib.util, pathlib
    _p = pathlib.Path(importlib.util.find_spec("chumpy").submodule_search_locations[0])
    for _f in _p.glob("*.py"):
        _s = _f.read_text(); _o = _s
        _s = _s.replace("from numpy import bool, int, float, complex, object, unicode, str, nan, inf", "from numpy import nan, inf")
        _s = _s.replace("inspect.getargspec", "inspect.getfullargspec")
        if _s != _o: _f.write_text(_s)
    # 2) our code from Drive (newest zip that contains run.py)
    from google.colab import drive
    drive.mount("/content/drive")
    D = "/content/drive/MyDrive/mytronD"
    zs = [z for z in glob.glob(f"{D}/*.zip") if any(n.endswith("ps4v4/pipeline.py") for n in zipfile.ZipFile(z).namelist())]
    assert zs, f"upload ps4v4.zip (the code) to {D}"
    Z = max(zs, key=os.path.getmtime)
    shutil.rmtree("/content/_ego_src", ignore_errors=True); zipfile.ZipFile(Z).extractall("/content/_ego_src")
    CODE = os.path.dirname(min(glob.glob("/content/_ego_src/**/run.py", recursive=True), key=len))
    print("code from", Z, "->", CODE)
    sh(f'cd "{CODE}" && python -c "from ps4v4.models.wilor_loader import load_wilor; load_wilor(\'cuda\'); print(\'WiLoR OK\')"')
    sh("python -c \"from geocalib import GeoCalib; GeoCalib(); print('GeoCalib OK')\"")
    sh("python -c \"from transformers import CLIPModel, CLIPProcessor as P; CLIPModel.from_pretrained('openai/clip-vit-base-patch32'); P.from_pretrained('openai/clip-vit-base-patch32'); print('CLIP OK')\"")
    # 3) EgoHands
    E = "/content/egohands"
    if not os.path.exists(f"{E}/metadata.mat"):
        os.makedirs(E, exist_ok=True)
        sh(f"wget -q -O {E}/ego.zip http://vision.soic.indiana.edu/egohands_files/egohands_data.zip")
        sh(f"cd {E} && unzip -q ego.zip && rm ego.zip")
    WORK = "/content/ego_work"
    EXTRA_SETS = []
else:
    CODE, E, WORK = os.environ["EGO_CODE"], os.environ["EGO_DATA"], os.environ["EGO_WORK"]
    EXTRA_SETS = ["device=cpu", "proposals.backend=fake", "probe.verifier=fake", "recon.backend=fake", "camera.use_geocalib=false"]

import numpy as np, cv2, scipy.io
mat = glob.glob(f"{E}/**/metadata.mat", recursive=True)[0]
root = os.path.dirname(mat)
md = scipy.io.loadmat(mat, squeeze_me=True, struct_as_record=False)
videos = list(np.atleast_1d(md["video"]))
videos = sorted(videos, key=lambda v: str(v.video_id))
step = max(1, len(videos) // N_VIDEOS)
videos = videos[::step][:N_VIDEOS]

def polys_to_box(p):
    p = np.asarray(p, float)
    if p.size < 6:
        return None
    p = p.reshape(-1, 2)
    return [p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max()]

# 4) 100 labelled frames -> one short clip per video; ground-truth boxes per clip frame
os.makedirs(f"{WORK}/clips", exist_ok=True)
gt = {}
for v in videos:
    vid = str(v.video_id)
    frames = sorted(np.atleast_1d(v.labelled_frames), key=lambda f: int(f.frame_num))
    folder = os.path.join(root, "_LABELLED_SAMPLES", vid)
    files = {int(re.findall(r"\d+", os.path.basename(p))[-1]): p for p in glob.glob(f"{folder}/*.jpg")}
    rows, imgs = [], []
    for f in frames:
        p = files.get(int(f.frame_num))
        if p is None:
            continue
        img = cv2.imread(p); imgs.append(img)
        H, W = img.shape[:2]
        r = {}
        for key in ("myleft", "myright", "yourleft", "yourright"):
            b = polys_to_box(getattr(f, key))
            if b is not None and (b[2] > W + 2 or b[3] > H + 2):          # stored as (y, x): swap
                b = [b[1], b[0], b[3], b[2]]
            r[key] = b
        rows.append(r)
    H, W = imgs[0].shape[:2]
    out = f"{WORK}/clips/{vid}.mp4"
    vw = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
    for im in imgs:
        vw.write(im)
    vw.release()
    gt[vid] = rows
print(f"{len(gt)} videos, {sum(len(r) for r in gt.values())} labelled frames")

# 5) our pipeline on each clip (sparse-frame settings: frames are ~1 s apart, so no flow/head tracking/infill)
SETS = [f"paths.cache_root={WORK}/out", "camera.geocalib_frames=3", "camera.head_rotation.enabled=false",
        "proposals.enhanced_pass=false", "tracklets.gate_dist=3.0", "tracklets.propagate=none",
        "tracklets.head_compensation=false", "probe.min_det=2", "probe.stride=1", "assign.min_det=2",
        "assign.min_piece=2", "assign.static_min_s=1000", "sequence.infill_max_s=0",
        "sequence.estimate_out_of_view=false", "render.enabled=false", "quality.enabled=false"] + EXTRA_SETS
from concurrent.futures import ThreadPoolExecutor
def run_one(vid):
    if os.path.exists(f"{WORK}/out/{vid}/hands.jsonl"):
        return vid, 0
    with open(f"{WORK}/{vid}.log", "w") as fh:
        rc = subprocess.run([sys.executable, "run.py", "--clip", f"{WORK}/clips/{vid}.mp4", "--set", *SETS],
                            cwd=CODE, stdout=fh, stderr=subprocess.STDOUT).returncode
    return vid, rc
with ThreadPoolExecutor(WORKERS) as ex:
    for vid, rc in ex.map(run_one, list(gt)):
        print(f"  {vid}: {'ok' if rc == 0 else 'FAILED (see ' + WORK + '/' + vid + '.log)'}", flush=True)

# 6) score: ours vs the raw detector (same model, no track-first decisions)
def iou(a, b):
    w = max(0, min(a[2], b[2]) - max(a[0], b[0])); h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - w * h
    return w * h / u if u > 0 else 0.0

def score(pred_by_frame):
    c = dict(own=0, own_found=0, side_ok=0, other=0, other_kept=0, preds=0, preds_own=0)
    for vid, rows in gt.items():
        for i, r in enumerate(rows):
            preds = pred_by_frame.get(vid, {}).get(i, [])
            c["preds"] += len(preds)
            own = [(k, r[k]) for k in ("myleft", "myright") if r[k] is not None]
            oth = [r[k] for k in ("yourleft", "yourright") if r[k] is not None]
            used = set()
            for k, b in own:                                           # greedy best-IoU matching
                c["own"] += 1
                best = max(((iou(b, p["box"]), j) for j, p in enumerate(preds) if j not in used), default=(0, -1))
                if best[0] >= IOU:
                    used.add(best[1]); c["own_found"] += 1; c["preds_own"] += 1
                    c["side_ok"] += preds[best[1]]["side"] == ("left" if k == "myleft" else "right")
            for b in oth:
                c["other"] += 1
                c["other_kept"] += any(iou(b, p["box"]) >= IOU for j, p in enumerate(preds) if j not in used)
    d = lambda a, b: a / b if b else float("nan")                      # noqa: E731
    return {"wearer-hand recall": d(c["own_found"], c["own"]), "left/right accuracy": d(c["side_ok"], c["own_found"]),
            "other-person hands rejected": 1 - d(c["other_kept"], c["other"]),
            "precision (tracked boxes that are the wearer's)": d(c["preds_own"], c["preds"]),
            "_counts": c}

ours, base = {}, {}
for vid in gt:
    o = f"{WORK}/out/{vid}"
    if not os.path.exists(f"{o}/hands.jsonl"):
        continue
    ours[vid] = {}
    for line in open(f"{o}/hands.jsonl"):
        r = json.loads(line)
        ours[vid][r["frame"]] = [{"box": h["bbox"], "side": h["side"]} for h in r["hands"]
                                 if h.get("source") in ("observed", "infilled") and "bbox" in h]
    props = pickle.load(open(f"{o}/proposals.pkl", "rb"))
    base[vid] = {i: [{"box": list(p[:4]), "side": "right" if p[5] > 0.5 else "left"} for p in a if p[4] >= 0.5]
                 for i, a in enumerate(props)}
gt = {k: v for k, v in gt.items() if k in ours}
R_ours, R_base = score(ours), score(base)
print(f"\nEgoHands accuracy: {len(gt)} videos, {sum(len(r) for r in gt.values())} labelled frames, "
      f"{R_ours['_counts']['own']} wearer hands, {R_ours['_counts']['other']} other-person hands (IoU >= {IOU})\n")
print(f"{'metric':52s} {'SpectatorAI':>12s} {'detector only':>14s}")
for k in R_ours:
    if not k.startswith("_"):
        print(f"{k:52s} {100 * R_ours[k]:11.1f}% {100 * R_base[k]:13.1f}%")
print(f"\n(detector only = the same WiLoR detector, boxes with score >= 0.5, its own left/right call; "
      f"no track-first decisions)\nNote: EgoHands labels ~1 frame per second, so tracking runs on sparse frames "
      f"(harder than continuous video). Total time {(time.time() - t_start) / 60:.0f} min.")
json.dump({"ours": R_ours, "detector_only": R_base, "videos": list(gt)},
          open(f"{WORK}/egohands_results.json", "w"), indent=1, default=float)
if not TEST:
    shutil.copy(f"{WORK}/egohands_results.json", f"{D}/egohands_results.json")
    print("saved", f"{D}/egohands_results.json")
